# Xerxes20 LightGBM challenger analysis

## tl;dr

Frozen-gate result: **`STOP_NO_SCOUT_CALIBRATION_WINNER`**. None of the four direct
`target_xerxes_20` LightGBM scouts cleared calibration eligibility. Every scout
missed the strict BMC Sharpe `> 0.20` requirement; `r1_trees2k` also missed the
BMC mean `> 0.0010` requirement. The highest calibration BMC mean was
`0.001659` from `r1_depth8`, with BMC Sharpe
`0.1751`.

Per the frozen stop rule, the locked final 50 scout eras were not scored, no
confirmation was run, and no model was packaged or uploaded.


## Context & Methods

This experiment tested four fixed LightGBM capacity profiles trained directly
on `target_xerxes_20` with Numerai's 780 medium features. Selection was based on
`target_ender_20` performance against `v53_lgbm_ender20`. The scout produced
1,279,658 OOF prediction rows across 214 retained eras, but the predeclared
selection decision used only the first 164 eras (`0373`-`1025`).

### Key Assumptions

- `gate.md`, the four configs, and their source/runtime receipts were frozen
  before scoring.
- The evaluator's final JSON and calibration CSVs are the authoritative outputs.
- This notebook verifies those content hashes and independently reconciles the
  saved calibration summaries; it does not retrain or rescore predictions.
- The prediction parquets and per-run result JSONs are intentionally not opened
  here because no scout earned access to the locked 50-era slice.


## Data & Integrity

In [1]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd

RESULT_JSON = Path('results/xerxes20_result.json').resolve()
RESULTS_DIR = RESULT_JSON.parent
EXPERIMENT_DIR = RESULT_JSON.parent.parent
REPO_ROOT = EXPERIMENT_DIR.parents[3]
EXPECTED_RESULT_SHA256 = 'c5939fc19c57688788fc2fdd2e28e8a49e99394ecab5aac019ddf1069cd62c6d'
EXPECTED_GENERATION_ID = '92389e16ab7f6fe244c1'
EXPECTED_CANDIDATES = ('r1_base_d6_t6000', 'r1_trees2k', 'r1_depth5', 'r1_depth8')

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

assert sys.version_info[:2] == (3, 12), sys.version
assert sha256_file(RESULT_JSON) == EXPECTED_RESULT_SHA256
result = json.loads(RESULT_JSON.read_text(encoding="utf-8"))
assert result["generation_id"] == EXPECTED_GENERATION_ID
assert result["state"] == "STOP_NO_SCOUT_CALIBRATION_WINNER"

content_result = RESULTS_DIR / f"xerxes20_result-{EXPECTED_GENERATION_ID}.json"
assert content_result.read_bytes() == RESULT_JSON.read_bytes()
print(f"Python {sys.version.split()[0]} | evaluator generation {EXPECTED_GENERATION_ID}")
print(f"Canonical result SHA-256: {EXPECTED_RESULT_SHA256}")


Python 3.12.13 | evaluator generation 92389e16ab7f6fe244c1
Canonical result SHA-256: c5939fc19c57688788fc2fdd2e28e8a49e99394ecab5aac019ddf1069cd62c6d


In [2]:
summary_path = RESULTS_DIR / Path(result["outputs"]["summary_csv"]).name
per_era_path = RESULTS_DIR / Path(result["outputs"]["per_era_csv"]).name

verified_files = [
    ("canonical result", RESULT_JSON, EXPECTED_RESULT_SHA256),
    ("content result", content_result, EXPECTED_RESULT_SHA256),
    ("summary CSV", summary_path, result["outputs"]["summary_csv_sha256"]),
    ("per-era CSV", per_era_path, result["outputs"]["per_era_csv_sha256"]),
]
for label, relative_path, expected_hash in [
    ("GPU runtime", result["inputs"]["gpu_runtime"]["path"], result["inputs"]["gpu_runtime"]["runtime_receipt_sha256"]),
    ("source manifest", result["inputs"]["source_manifest"]["path"], result["inputs"]["source_manifest"]["sha256"]),
    ("evaluator", result["evaluator"]["path"], result["evaluator"]["sha256"]),
]:
    verified_files.append((label, REPO_ROOT / relative_path, expected_hash))
for candidate in EXPECTED_CANDIDATES:
    verified_files.append((
        f"config {candidate}",
        EXPERIMENT_DIR / "configs" / f"{candidate}.py",
        result["inputs"]["scout_runs"][candidate]["config_sha256"],
    ))

verification_rows = []
for label, path, expected_hash in verified_files:
    actual_hash = sha256_file(path)
    assert actual_hash == expected_hash, label
    verification_rows.append({"artifact": label, "sha256": actual_hash})

print(pd.DataFrame(verification_rows).to_string(index=False))
print("\nPrediction and per-run result files: evaluator receipts retained; files not opened.")


               artifact                                                           sha256
       canonical result c5939fc19c57688788fc2fdd2e28e8a49e99394ecab5aac019ddf1069cd62c6d
         content result c5939fc19c57688788fc2fdd2e28e8a49e99394ecab5aac019ddf1069cd62c6d
            summary CSV 9eacdcdcf5a94f9b9fc0194173efd6588fe5cf37588bda77ff2690273c3fa862
            per-era CSV 41259804c93fca8d57ed6eeb72bf597e4ce558ac14dfea6223a1e4baa89b4cf2
            GPU runtime d6656bc39a0d603860c9b327569bd453b1556b8a3aae99f8567edefbc214f135
        source manifest 4b3dd7e30dbcb8e532ffbdd484031c98efc30cf4d82804290f62846e19675a8d
              evaluator d6b3e0290a28e4fbd12376227f2dcbda21defe6263bb595e51042c139097568c
config r1_base_d6_t6000 5ec02a6647eb6f14dea6fc3a7c8c358f9fd69e3e37dcd4220d41c25c3fbf143b
      config r1_trees2k b1211a94b72f5f8236d4d9d3a4e82e6b5e7cdc2df9b8f4b8fcd746e753056c2a
       config r1_depth5 05205e3852984db5b1ed09a65ecc19bf4e376e845bc2c3d59c395a8b1da3deb5
       config r1_dept

## Results

In [3]:
summary = pd.read_csv(summary_path)
per_era = pd.read_csv(per_era_path, dtype={"era": str})

assert set(per_era["phase"]) == {"scout_calibration"}
assert set(per_era["candidate"]) == set(EXPECTED_CANDIDATES)
assert not per_era.duplicated(["phase", "candidate", "era"]).any()
assert np.isfinite(per_era[["corr", "bmc", "benchmark_similarity"]].to_numpy()).all()
assert len(per_era) == 4 * 164

reference_eras = None
recomputed_rows = []
for candidate in EXPECTED_CANDIDATES:
    candidate_rows = per_era.loc[per_era["candidate"] == candidate].sort_values("era")
    eras = candidate_rows["era"].tolist()
    assert len(eras) == 164 and eras[0] == "0373" and eras[-1] == "1025"
    if reference_eras is None:
        reference_eras = eras
    else:
        assert eras == reference_eras
    bmc = candidate_rows["bmc"]
    corr = candidate_rows["corr"]
    cumulative_bmc = bmc.cumsum()
    recomputed_rows.append({
        "candidate": candidate,
        "era_count": len(candidate_rows),
        "corr_mean": corr.mean(),
        "corr_std": corr.std(ddof=0),
        "corr_sharpe": corr.mean() / corr.std(ddof=0),
        "bmc_mean": bmc.mean(),
        "bmc_std": bmc.std(ddof=0),
        "bmc_sharpe": bmc.mean() / bmc.std(ddof=0),
        "bmc_max_drawdown": (cumulative_bmc.cummax() - cumulative_bmc).max(),
        "avg_benchmark_similarity": candidate_rows["benchmark_similarity"].mean(),
    })

recomputed = pd.DataFrame(recomputed_rows).set_index("candidate")
reported = summary.set_index("candidate").loc[list(EXPECTED_CANDIDATES)]
metric_columns = [
    "era_count", "corr_mean", "corr_std", "corr_sharpe", "bmc_mean",
    "bmc_std", "bmc_sharpe", "bmc_max_drawdown", "avg_benchmark_similarity",
]
assert np.allclose(
    reported[metric_columns].to_numpy(dtype=float),
    recomputed[metric_columns].to_numpy(dtype=float),
    rtol=0.0,
    atol=1e-12,
)
print("656 calibration rows reconcile to the evaluator summary within 1e-12.")


656 calibration rows reconcile to the evaluator summary within 1e-12.


In [4]:
thresholds = result["thresholds"]["scout_calibration"]
decision_rows = []
for candidate in EXPECTED_CANDIDATES:
    metrics = reported.loc[candidate]
    independent_checks = {
        "bmc_mean": metrics["bmc_mean"] > thresholds["bmc_mean_min_exclusive"],
        "bmc_sharpe": metrics["bmc_sharpe"] > thresholds["bmc_sharpe_min_exclusive"],
        "bmc_max_drawdown": metrics["bmc_max_drawdown"] < thresholds["bmc_max_drawdown_max_exclusive"],
        "corr_mean": metrics["corr_mean"] > thresholds["corr_mean_min_exclusive"],
        "benchmark_similarity": metrics["avg_benchmark_similarity"] < thresholds["benchmark_similarity_max_exclusive"],
    }
    recorded = result["scout_calibration_candidates"][candidate]
    assert independent_checks == recorded["checks"]
    assert recorded["eligible"] == all(independent_checks.values())
    failed = [name for name, passed in independent_checks.items() if not passed]
    decision_rows.append({
        "candidate": candidate,
        "BMC mean": metrics["bmc_mean"],
        "BMC Sharpe": metrics["bmc_sharpe"],
        "BMC max DD": metrics["bmc_max_drawdown"],
        "Ender Corr": metrics["corr_mean"],
        "benchmark Spearman": metrics["avg_benchmark_similarity"],
        "failed checks": ", ".join(failed),
    })

decision_table = pd.DataFrame(decision_rows)
numeric_columns = ["BMC mean", "BMC Sharpe", "BMC max DD", "Ender Corr", "benchmark Spearman"]
decision_table[numeric_columns] = decision_table[numeric_columns].round(6)
print(decision_table.to_string(index=False))

assert not any(item["eligible"] for item in result["scout_calibration_candidates"].values())
assert result["selected_scout"] is None
assert result["scout_holdout_checks"] == {}
assert result["confirmation_checks"] == {}
assert result["offline_gate_passed"] is False
assert result["promotion_eligible"] is False
print(f"\nFinal decision: {result['state']}")


       candidate  BMC mean  BMC Sharpe  BMC max DD  Ender Corr  benchmark Spearman        failed checks
r1_base_d6_t6000  0.001613    0.168329    0.042231    0.021567            0.451152           bmc_sharpe
      r1_trees2k  0.000402    0.042282    0.069234    0.017667            0.392992 bmc_mean, bmc_sharpe
       r1_depth5  0.001225    0.128079    0.048439    0.020754            0.441641           bmc_sharpe
       r1_depth8  0.001659    0.175127    0.042528    0.021912            0.458295           bmc_sharpe

Final decision: STOP_NO_SCOUT_CALIBRATION_WINNER


## Takeaways

`r1_depth8` had the strongest calibration BMC mean and Ender Corr, but its BMC
Sharpe was `0.1751`, below the frozen strict `> 0.20` gate. The other three
scouts also failed BMC Sharpe; the 2,000-tree scout additionally failed minimum
BMC mean. Capacity changed signal strength, but none produced sufficiently
stable unique signal under the predeclared screen.

The correct decision is **not promotion eligible**. The experiment stops at
calibration without opening the locked 50-era metrics, running a consecutive
confirmation, packaging a pickle, uploading a model, submitting predictions,
or staking.
